In [10]:
import pandas as pd
from sklearn.cluster import KMeans

In [11]:
mov_h = pd.read_csv("/Users/fangsiyu/Desktop/Kolding Hackathon 2026/n_data/movement_stats_hourly.csv")
sd_h = pd.read_csv("/Users/fangsiyu/Desktop/Kolding Hackathon 2026/n_data/stay_duration_stats_hourly.csv")

mov_h['sensor'] = mov_h['sensor'].str.extract(r'(\d+\.\d+)')

mov_h
sd_h

,Unnamed: 0,sensor,category,sum_ms,min_ms,max_ms,objects,timestamp
0,0,12-Zone 1 - Duration of occurrence - Table,wheelchair,21360,6240,15120,2,2026-02-09T15:00:00+01:00
1,1,12-Zone 1 - Duration of occurrence - Table,van,535600,8560,329360,4,2026-02-09T15:00:00+01:00
2,2,12-Zone 1 - Duration of occurrence - Table,scooter,16040,6800,9240,2,2026-02-09T15:00:00+01:00
3,3,12-Zone 1 - Duration of occurrence - Table,pram,143680,3040,18120,22,2026-02-09T15:00:00+01:00
4,4,12-Zone 1 - Duration of occurrence - Table,pedestrian,7314036,3080,597560,409,2026-02-09T15:00:00+01:00
...,...,...,...,...,...,...,...,...
27753,4377,12-Zone 1 - Duration of occurrence - Table,pedestrian,81760,3160,8520,16,2026-05-08T07:00:00+02:00
27754,4378,12-Zone 1 - Duration of occurrence - Table,car trailer,3160,3160,3160,1,2026-05-08T07:00:00+02:00
27755,4379,12-Zone 1 - Duration of occurrence - Table,car,551719,3080,8440,112,2026-05-08T07:00:00+02:00
27756,4380,12-Zone 1 - Duration of occurrence - Table,bicycle,3040,3040,3040,1,2026-05-08T07:00:00+02:00


In [12]:
mov_h_sensor_grouped = mov_h.groupby(['timestamp', 'sensor', 'category', 'direction'])['amount'].sum().unstack(fill_value=0)

mov_h_sensor_grouped['net_inflow'] = mov_h_sensor_grouped['IN'] - mov_h_sensor_grouped['OUT']

mov_h_sensor_grouped = mov_h_sensor_grouped.reset_index()
mov_h_sensor_grouped.columns.name = None
# mov_h_sensor_grouped.to_csv('/Users/fangsiyu/Desktop/Kolding Hackathon 2026/mov_h_net_flow_by_gate.csv', index=False, encoding='utf-8-sig')
mov_h_sensor_grouped

,timestamp,sensor,category,IN,OUT,net_inflow
0,2026-02-09T15:00:00+01:00,1.1,bicycle,3,1,2
1,2026-02-09T15:00:00+01:00,1.1,car,0,1,-1
2,2026-02-09T15:00:00+01:00,1.1,pedestrian,31,39,-8
3,2026-02-09T15:00:00+01:00,1.1,scooter,0,1,-1
4,2026-02-09T15:00:00+01:00,1.2,bicycle,2,2,0
...,...,...,...,...,...,...
23833,2026-05-08T08:00:00+02:00,2.2,light,1,0,1
23834,2026-05-08T08:00:00+02:00,3.1,animal,0,1,-1
23835,2026-05-08T08:00:00+02:00,3.1,bicycle,0,1,-1
23836,2026-05-08T08:00:00+02:00,3.1,car,3,3,0


In [13]:
mask = mov_h['sensor'] != "3.1"
n_mov = mov_h[mask]

grouped_whole_zone = n_mov.groupby(['timestamp', 'category', 'direction'])['amount'].sum().unstack(fill_value=0)

grouped_whole_zone['net_inflow'] = grouped_whole_zone['IN'] - grouped_whole_zone['OUT']

grouped_whole_zone = grouped_whole_zone.reset_index()
grouped_whole_zone.columns.name = None
# grouped_whole_zone.to_csv('/Users/fangsiyu/Desktop/Kolding Hackathon 2026/mov_h_net_flow_by_mix_gate_without3-1.csv', index=False, encoding='utf-8-sig')
grouped_whole_zone

,timestamp,category,IN,OUT,net_inflow
0,2026-02-09T15:00:00+01:00,bicycle,5,3,2
1,2026-02-09T15:00:00+01:00,car,13,13,0
2,2026-02-09T15:00:00+01:00,motorcycle,0,1,-1
3,2026-02-09T15:00:00+01:00,pedestrian,131,163,-32
4,2026-02-09T15:00:00+01:00,scooter,0,1,-1
...,...,...,...,...,...
8717,2026-05-08T07:00:00+02:00,pedestrian,4,4,0
8718,2026-05-08T07:00:00+02:00,tractor,1,0,1
8719,2026-05-08T07:00:00+02:00,van,2,0,2
8720,2026-05-08T08:00:00+02:00,car,0,1,-1


In [14]:
mov_p = grouped_whole_zone.pivot(index='timestamp', columns='category', values='net_inflow').fillna(0)

mov_p = mov_p.reset_index()
mov_p.columns.name = None

mov_p = mov_p.add_prefix('mov_net_')
mov_p.columns.name = None


mov_p

,mov_net_timestamp,mov_net_animal,mov_net_bicycle,mov_net_bus,mov_net_car,mov_net_car trailer,mov_net_heavy,mov_net_light,mov_net_motorcycle,mov_net_pedestrian,mov_net_pram,mov_net_scooter,mov_net_tractor,mov_net_tram,mov_net_truck trailer,mov_net_van,mov_net_wheelchair
0,2026-02-09T15:00:00+01:00,0.0,2.0,0.0,0.0,0.0,0.0,0.0,-1.0,-32.0,0.0,-1.0,0.0,0.0,0.0,0.0,0.0
1,2026-02-09T16:00:00+01:00,0.0,-5.0,0.0,-9.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,-1.0,0.0
2,2026-02-09T17:00:00+01:00,1.0,1.0,0.0,-1.0,0.0,0.0,0.0,0.0,18.0,-1.0,0.0,0.0,0.0,0.0,0.0,0.0
3,2026-02-09T18:00:00+01:00,-5.0,-1.0,0.0,0.0,0.0,0.0,0.0,0.0,21.0,1.0,0.0,0.0,0.0,0.0,-1.0,0.0
4,2026-02-09T19:00:00+01:00,0.0,-1.0,0.0,-5.0,0.0,0.0,0.0,0.0,-20.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2035,2026-05-08T03:00:00+02:00,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2036,2026-05-08T05:00:00+02:00,0.0,-1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2037,2026-05-08T06:00:00+02:00,0.0,0.0,0.0,3.0,0.0,0.0,0.0,0.0,-1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2038,2026-05-08T07:00:00+02:00,0.0,2.0,0.0,-4.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,2.0,0.0


In [15]:
sd_mask = sd_h['sensor'] == '20-Zone 2 - Time of occurrence - Table'
zone1_df = sd_h[sd_mask].copy()

pivot_df = zone1_df.pivot_table(
    index='timestamp', 
    columns='category', 
    values= ['sum_ms', 'min_ms', 'max_ms'], 
    aggfunc={
        'sum_ms': 'sum',
        'min_ms': 'min',
        'max_ms': 'max'
    }
).fillna(0)

pivot_df.columns = [f"dur_{metric}_{category}" for metric, category in pivot_df.columns]
final_zone1_all_metrics = pivot_df.reset_index()
final_zone1_all_metrics.to_csv('/Users/fangsiyu/Desktop/Kolding Hackathon 2026/dur_h_zone2_pivot.csv', index=False, encoding='utf-8-sig')

final_zone1_all_metrics

,timestamp,dur_max_ms_animal,dur_max_ms_bicycle,dur_max_ms_bus,dur_max_ms_car,dur_max_ms_car trailer,dur_max_ms_caravan,dur_max_ms_heavy,dur_max_ms_light,dur_max_ms_motorcycle,...,dur_sum_ms_light,dur_sum_ms_motorcycle,dur_sum_ms_pedestrian,dur_sum_ms_pram,dur_sum_ms_scooter,dur_sum_ms_tractor,dur_sum_ms_tram,dur_sum_ms_truck trailer,dur_sum_ms_van,dur_sum_ms_wheelchair
0,2026-02-10T11:00:00+01:00,2000.0,3520.0,0.0,0.0,0.0,0.0,0.0,7760.0,0.0,...,7760.0,0.0,253639.0,33600.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2026-02-10T12:00:00+01:00,24600.0,4520.0,0.0,440.0,0.0,0.0,0.0,48760.0,4520.0,...,526879.0,7640.0,1087969.0,112610.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2026-02-10T13:00:00+01:00,3960.0,2760.0,0.0,160.0,0.0,0.0,0.0,48760.0,11920.0,...,832980.0,22320.0,2576213.0,69430.0,400.0,0.0,17240.0,0.0,10620.0,0.0
3,2026-02-10T14:00:00+01:00,13880.0,0.0,0.0,4200.0,47920.0,0.0,0.0,33240.0,5480.0,...,195480.0,8360.0,2751555.0,80560.0,0.0,0.0,4690.0,0.0,5960.0,0.0
4,2026-02-10T15:00:00+01:00,18000.0,5440.0,0.0,3280.0,42480.0,0.0,0.0,28680.0,480.0,...,90590.0,480.0,1423226.0,26640.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2076,2026-05-08T03:00:00+02:00,7240.0,0.0,0.0,320.0,120.0,0.0,0.0,0.0,520.0,...,0.0,520.0,149240.0,45530.0,1040.0,0.0,0.0,0.0,0.0,0.0
2077,2026-05-08T04:00:00+02:00,3160.0,2000.0,0.0,8000.0,0.0,0.0,0.0,0.0,8000.0,...,0.0,50120.0,74560.0,0.0,1560.0,0.0,0.0,0.0,0.0,0.0
2078,2026-05-08T05:00:00+02:00,2000.0,560.0,0.0,8000.0,0.0,0.0,0.0,1040.0,8000.0,...,1040.0,113480.0,41920.0,0.0,9800.0,0.0,0.0,0.0,0.0,0.0
2079,2026-05-08T06:00:00+02:00,2000.0,2200.0,0.0,7760.0,0.0,0.0,0.0,120.0,2040.0,...,120.0,12040.0,52880.0,520.0,0.0,0.0,0.0,0.0,1760.0,0.0


In [ ]:
temp_df = pd.merge(final_zone1_all_metrics, mov_p, left_on='timestamp', right_on='mov_net_timestamp',how='left')
# tt_df = temp_df[temp_df.isna().any(axis=1)].head(66)
# tt_df.to_csv("/Users/fangsiyu/Desktop/Kolding Hackathon 2026/what???/ttdf.csv", index=False, encoding='utf-8-sig')

X = temp_df.drop(columns=['timestamp','mov_net_timestamp'])
X = X.fillna(0)

kmeans = KMeans(n_clusters=3, random_state=42)

temp_df['cluster_label'] = kmeans.fit_predict(X)
temp_df[['timestamp','cluster_label']]

temp_df.to_csv('/Users/fangsiyu/Desktop/Kolding Hackathon 2026/cluster_h_kmean_k3.csv', index=False, encoding='utf-8-sig')